### 전처리: 여백 제거
- 전형적인 컴퓨터 비전 기반의 전처리에서 아이디어

In [ ]:
import cv2
import numpy as np

def detect_boundary_with_5px_window(image_path):
    image = cv2.imread(image_path)
    if image is None: return None
    h, w = image.shape[:2]
    
    # 분석을 위해 Gray로 변환 (색상 변화량 측정 용이)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    def get_diff(p1, p2):
        return abs(int(p1) - int(p2))

    # 결과 좌표 저장 (초기값은 이미지 끝단)
    top, bottom, left, right = 0, h-1, 0, w-1
    win_size = 5 # 연구원님이 제안하신 5px 윈도우

    # --- 하단 스캔 (Bottom-up) ---
    for y in range(h - win_size - 1, int(h * 0.3), -1):
        # y지점과 y-5지점의 중앙부 평균 밝기 비교
        # 5px 구간의 '평균'을 비교하여 노이즈 상쇄
        current_win = np.mean(gray[y : y + win_size, int(w*0.4):int(w*0.6)])
        next_win = np.mean(gray[y - win_size : y, int(w*0.4):int(w*0.6)])
        
        # 5px 이동 시 밝기 변화량이 15(임계값) 이상이면 경계로 간주
        if get_diff(current_win, next_win) > 15:
            bottom = y
            break

    # --- 상단 스캔 (Top-down) ---
    for y in range(win_size, int(h * 0.4)):
        current_win = np.mean(gray[y - win_size : y, int(w*0.4):int(w*0.6)])
        next_win = np.mean(gray[y : y + win_size, int(w*0.4):int(w*0.6)])
        if get_diff(current_win, next_win) > 15:
            top = y
            break

    # --- 좌측 스캔 (Left-to-Right) ---
    for x in range(win_size, int(w * 0.2)):
        current_win = np.mean(gray[int(h*0.4):int(h*0.6), x - win_size : x])
        next_win = np.mean(gray[int(h*0.4):int(h*0.6), x : x + win_size])
        if get_diff(current_win, next_win) > 15:
            left = x
            break

    # --- 우측 스캔 (Right-to-Left) ---
    for x in range(w - win_size - 1, int(w * 0.8), -1):
        current_win = np.mean(gray[int(h*0.4):int(h*0.6), x : x + win_size])
        next_win = np.mean(gray[int(h*0.4):int(h*0.6), x - win_size : x])
        if get_diff(current_win, next_win) > 15:
            right = x
            break

    # 결과 크롭 및 저장
    cropped = image[top:bottom, left:right]
    cv2.imwrite("result_5px_window.jpg", cropped)
    print(f"5px 윈도우 스캔 완료: T:{top}, B:{bottom}, L:{left}, R:{right}")
    
    return cropped

# 실행
detect_boundary_with_5px_window('./KakaoTalk_20260322_185520413.jpg')

5px 윈도우 스캔 완료: T:474, B:3631, L:6, R:2168


array([[[ 88,  84,  83],
        [ 89,  85,  84],
        [ 94,  90,  89],
        ...,
        [141, 152, 166],
        [150, 162, 174],
        [152, 164, 176]],

       [[ 83,  79,  78],
        [ 86,  82,  81],
        [ 88,  84,  83],
        ...,
        [145, 156, 170],
        [149, 161, 173],
        [152, 164, 176]],

       [[ 73,  69,  68],
        [ 83,  79,  78],
        [ 86,  82,  81],
        ...,
        [154, 167, 181],
        [155, 167, 179],
        [152, 164, 176]],

       ...,

       [[123, 133, 150],
        [118, 128, 145],
        [113, 123, 140],
        ...,
        [185, 176, 172],
        [186, 177, 173],
        [182, 173, 169]],

       [[112, 122, 139],
        [110, 120, 137],
        [106, 116, 133],
        ...,
        [189, 180, 176],
        [185, 176, 172],
        [179, 170, 166]],

       [[107, 117, 135],
        [107, 117, 135],
        [105, 115, 133],
        ...,
        [188, 179, 175],
        [183, 174, 170],
        [181, 172, 168]]

① 노이즈 제거 및 선명화 실행 코드  
이 코드는 배경의 노이즈를 밀어버리고 글자의 대비를 극대화하여, 이후 영역 분할(ROI) 시 텍스트 경계를 명확히 잡을 수 있게 도와줍니다.

In [120]:
import cv2
import numpy as np

def preprocess_for_ocr(image_path):
    # 1. 이미지 로드
    img = cv2.imread(image_path)
    if img is None: return print("이미지를 로드할 수 없습니다.")
    
    # 2. 그레이스케일 변환
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 3. 조명 보정 및 노이즈 제거 (가우시안 블러)
    # 너무 강하게 주면 작은 글씨가 뭉개지므로 (3,3) 정도가 적당합니다.
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    
    # 4. 적응형 이진화 (핵심: 구겨진 종이의 그림자 극복)
    # cv2.ADAPTIVE_THRESH_GAUSSIAN_C를 사용하여 지역적 명암 차이를 극복합니다.
    # blockSize(15)와 C(2) 값은 이미지 상태에 따라 미세 조정 가능합니다.
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY, 15, 2)
    
    # 5. 모폴로지 연산 (글자 획 보강)
    # 이진화 과정에서 얇아진 수기 획을 살짝 두껍게 만들어 연결성을 높입니다.
    kernel = np.ones((2, 2), np.uint8)
    # 침식(Erode) 연산은 검은색 글자 영역에서는 글자를 두껍게 만드는 효과가 있습니다.
    # (이미지가 반전된 상태가 아니므로 가독성을 위해 살짝만 적용)
    processed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    
    # 6. 언샤프 마스킹 (선명화)
    # 글자 경계를 더 날카롭게 세워 Gemini가 획을 명확히 인지하게 합니다.
    gaussian = cv2.GaussianBlur(processed, (0, 0), 2.0)
    sharpened = cv2.addWeighted(processed, 1.5, gaussian, -0.5, 0)
    
    # 결과 저장
    cv2.imwrite("preprocessed_ocr_input.jpg", sharpened)
    print("선명화 처리가 완료되었습니다: 'preprocessed_ocr_input.jpg'")
    return sharpened

# 실행
input_file = 'result_5px_window.jpg' # 앞서 정렬된 이미지 사용
preprocess_for_ocr(input_file)

선명화 처리가 완료되었습니다: 'preprocessed_ocr_input.jpg'


array([[255, 255, 255, ...,   0,   0,   0],
       [255, 255, 255, ...,   0,   0,   0],
       [  0,   0, 255, ...,   0,   0,   0],
       ...,
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255]],
      shape=(3157, 2162), dtype=uint8)

### 위에서부터 자르기

1. 전처리 및 선 검출 (HoughLinesP)
* Canny(gray, 50, 150): 엣지를 검출하여 선의 후보를 찾음
* minLineLength=int(w * 0.25): 이미지 너비의 25% 이상 되는 선만 검출함 → 표의 테두리·구분선 등 명확한 직선만 유지하고, 수기 글씨나 작은 노이즈는 제거함

2. 가로 경계선 탐색 로직 (Shimstock 하단선)
* y_sep_memo = int(h * 0.35): 선을 못 찾았을 때 사용하는 기본값 (기존 0.45 → 0.35로 조정) → Shimstock 표가 위쪽에 있거나, 수기 메모 시작 위치가 더 위일 경우 대비한 설정
* if 0.35 * h < y1 < 0.45 * h: 선 탐색 수직 범위를 35%~45%로 제한함

3. 영역 분할 로직 (ROI Extraction)
* roi_rest_img[0:y_sep_memo, :]: 상단 표 영역 전체 추출
* roi_rest_img[y_sep_memo:, :]: 하단 영역 전체를 수기 메모(roi_3_memo)로 할당

4. 표 내부 정밀 분할 (Tooth vs Shimstock)
* x_sep = int(w * 0.53): 이미지 오른쪽 53% 지점부터 표 시작으로 설정
* y_tooth_sep = int(tables_h * 0.45): 표 영역 내 상단 45%를 치식 표, 하단 55%를 Shimstock 표로 분할



In [27]:
import cv2
import numpy as np

def split_stage_1(image_path):
    img = cv2.imread(image_path)
    if img is None: return None, None
    h, w = img.shape[:2]
    
    # 1. 로고 바 검출 (이전 로직 동일)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    logo_bar_y2 = 0
    if cnts:
        for c in cnts:
            x, y, cw, ch = cv2.boundingRect(c)
            if y < h * 0.15 and cw > w * 0.6:
                logo_bar_y2 = y + ch
                break
    
    # 2. 0.135 오프셋 적용하여 절단점 계산
    split_y = logo_bar_y2 + int(h * 0.135) if logo_bar_y2 > 0 else int(h * 0.28)
    
    # 3. 이미지 분할
    roi_1_patient = img[0:split_y, :]  # 환자 정보 영역
    roi_rest = img[split_y:, :]        # 나머지 전체 영역 (치식, 메모 등 포함)
    
    # 4. 저장 및 반환
    cv2.imwrite("roi_1_patient.jpg", roi_1_patient)
    cv2.imwrite("roi_rest.jpg", roi_rest)
    print(f"1차 분할 완료: 환자 정보 영역(0~{split_y}), 나머지 영역({split_y}~{h})")
    
    return roi_1_patient, roi_rest

# 실행
roi_1, roi_rest_img = split_stage_1('preprocessed_ocr_input.jpg')

1차 분할 완료: 환자 정보 영역(0~841), 나머지 영역(841~3305)


In [40]:
import cv2
import numpy as np

def split_stage_2_pure_crop(roi_rest_img):
    if roi_rest_img is None: return print("이미지를 로드할 수 없습니다.")
    h, w = roi_rest_img.shape[:2]
    
    # 1. 전처리 (에지 강조)
    gray = cv2.cvtColor(roi_rest_img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    
    # 2. 치식/Shimstock 표와 하단 수기 메모 사이의 경계선 찾기
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, 
                            minLineLength=int(w * 0.25), maxLineGap=12)
    
    y_sep_memo = int(h * 0.35) # Shimstock 표 하단 (메모 시작점) 기본값

    if lines is not None:
        # y축 기준 35%~60% 사이에서 발견되는 가장 윗쪽 가로선을 경계로 설정
        # 이 선이 '2번째 네모(Shimstock 표)'의 밑선이 됩니다.
        lines = sorted(lines, key=lambda x: x[0][1])
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if 0.35 * h < y1 < 0.45 * h and abs(x2 - x1) > w * 0.25:
                # 치식/Shimstock 표는 우측에 있으므로 우측에 치우친 선 선호
                if x1 > w * 0.4:
                    y_sep_memo = y1
                    break

    print(f"2차 분할점(y): {y_sep_memo}")

    # 3. 1단계: 가로 분할 (y 좌표)
    # 상단: 표 영역 전체 (roi_2_tables_area)
    # 하단: 수기 메모 영역 전체 (roi_3_memo)
    # 원본 이미지를 훼손하지 않고 좌표만으로 나눕니다.
    roi_tables_area = roi_rest_img[0:y_sep_memo, :]
    roi_3_memo = roi_rest_img[y_sep_memo:, :]
    
    # 4. 2단계: 표 영역 내부 정밀 분할 (x 좌표)
    # 우측 53% 지점 (x_sep)
    x_sep = int(w * 0.53)
    
    # ROI 2-a: 치식 표 (우측 상단 전체)
    # 여기서 표 영역 내부 가로 분할 (tables_h * 0.38 지점)
    tables_h = roi_tables_area.shape[0]
    y_tooth_sep = int(tables_h * 0.45)
    
    # 우측 상단 조각: 치식 표
    roi_2_a_tooth = roi_tables_area[0:y_tooth_sep, x_sep:]
    
    # ROI 2-b: Shimstock 표 (우측 하단 조각)
    roi_2_b_shimstock = roi_tables_area[y_tooth_sep:, x_sep:]
    
    # 5. 최종 저장
    cv2.imwrite("roi_2_a_tooth.jpg", roi_2_a_tooth)
    cv2.imwrite("roi_2_b_shimstock.jpg", roi_2_b_shimstock)
    cv2.imwrite("roi_3_memo.jpg", roi_3_memo)
    
    # 디버깅: 절단선 확인
    debug_img = roi_rest_img.copy()
    cv2.line(debug_img, (0, y_sep_memo), (w, y_sep_memo), (0, 0, 255), 3) # 메모 경계
    cv2.line(debug_img, (x_sep, 0), (x_sep, y_sep_memo), (255, 0, 0), 3) # 표 세로 경계
    cv2.line(debug_img, (x_sep, y_tooth_sep), (w, y_tooth_sep), (0, 255, 0), 2) # 표 내부 경계
    cv2.imwrite("debug_split_safe.jpg", debug_img)
    
    print("비파괴적 3개 영역 분할 저장 완료 (Tooth, Shimstock, Memo)")

# 실행: 앞서 자르고 남은 이미지('roi_rest.jpg')를 사용하여 실행
split_stage_2_pure_crop(roi_rest_img)

2차 분할점(y): 862
비파괴적 3개 영역 분할 저장 완료 (Tooth, Shimstock, Memo)


### 수기영역부터 분리하기

1단계: 로고 기반 메모 및 상단 영역 분리
로고 바 하단 끝에서 메모가 시작되는 지점까지의 오프셋을 활용합니다.

In [121]:
import cv2
import numpy as np

def split_memo_first(image_path):
    img = cv2.imread(image_path)
    if img is None: return None, None
    h, w = img.shape[:2]
    
    # 1. 로고 바 검출 (검은색 띠)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    logo_bar_y2 = 0
    if cnts:
        for c in cnts:
            x, y, cw, ch = cv2.boundingRect(c)
            if y < h * 0.15 and cw > w * 0.6:
                logo_bar_y2 = y + ch
                break

    # 2. 로고 바를 기준으로 메모 분리 지점(split_y_memo) 계산
    # '의뢰내용'이라는 인쇄물 근처를 기준으로 자릅니다.
    # 실험적으로 로고 끝에서 전체 높이의 약 35~40% 지점이 적당합니다.
    split_y_memo = logo_bar_y2 + int(h * 0.38) if logo_bar_y2 > 0 else int(h * 0.5)
    
    # 3. 영역 분리
    roi_upper = img[0:split_y_memo, :]  # 환자정보 + 치식/Shimstock 표 포함
    roi_memo = img[split_y_memo:, :]    # 하단 수기 메모 전체
    
    cv2.imwrite("roi_upper_area.jpg", roi_upper)
    cv2.imwrite("roi_3_memo.jpg", roi_memo)
    
    print(f"1차 분할 완료: 상단 영역(0~{split_y_memo}), 메모 영역({split_y_memo}~{h})")
    return roi_upper, roi_memo

# 실행
roi_upper, roi_memo = split_memo_first('preprocessed_ocr_input.jpg')

1차 분할 완료: 상단 영역(0~1529), 메모 영역(1529~3157)


### [로고 분리 → 좌우 수직 분할 → 좌측 가로 분리 → 우측 3단 분리]

In [122]:
import cv2
import numpy as np

def split_upper_precision(roi_upper_img):
    if roi_upper_img is None: 
        return print("이미지를 로드할 수 없습니다.")
    
    uh, uw = roi_upper_img.shape[:2]
    
    # 1. 로고 부분 자르기 (상단 약 22% 지점)
    # 이미지 최상단의 검은색 띠(로고 및 병원 정보) 영역을 분리합니다.
    y_logo_end = int(uh * 0.22)
    roi_logo = roi_upper_img[0:y_logo_end, :]
    
    # 로고를 제외한 나머지 콘텐츠 영역
    roi_content_area = roi_upper_img[y_logo_end:, :]
    ch, cw = roi_content_area.shape[:2]
    
    # 2. 로고 밑부분 세로로 반 자르기 (x축 기준 53% 지점)
    # 왼쪽(텍스트 정보)과 오른쪽(표 데이터)의 레이아웃이 다르므로 수직 분할합니다.
    x_mid = int(cw * 0.53)
    roi_left_col = roi_content_area[:, 0:x_mid]
    roi_right_col = roi_content_area[:, x_mid:]
    
    # 3. 왼쪽 영역 가로 분할 (Shade 글자 윗부분 기준)
    # 환자명/보철종류 영역과 Shade 전용 영역을 분리합니다.
    y_left_sep = int(ch * 0.45)
    roi_left_1_patient = roi_left_col[0:y_left_sep, :]
    roi_left_2_shade = roi_left_col[y_left_sep:, :]
    
    # 4. 오른쪽 영역 3분할 (치식 상자 기준)
    # 치식 상자(Tooth Chart)를 중심으로 위(날짜), 아래(Shimstock)를 나눕니다.
    y_right_top_sep = int(ch * 0.35)    # 치식 상자 윗부분 시작점
    y_right_bottom_sep = int(ch * 0.68) # 치식 상자 아랫부분 끝점
    
    roi_right_1_dates = roi_right_col[0:y_right_top_sep, :]
    roi_right_2_tooth = roi_right_col[y_right_top_sep:y_right_bottom_sep, :]
    roi_right_3_shimstock = roi_right_col[y_right_bottom_sep:, :]

    # 5. 결과 저장
    cv2.imwrite("roi_upper_0_logo.jpg", roi_logo)
    cv2.imwrite("roi_upper_1_left_patient.jpg", roi_left_1_patient)
    cv2.imwrite("roi_upper_2_left_shade.jpg", roi_left_2_shade)
    cv2.imwrite("roi_upper_3_right_dates.jpg", roi_right_1_dates)
    cv2.imwrite("roi_upper_4_right_tooth.jpg", roi_right_2_tooth)
    cv2.imwrite("roi_upper_5_right_shimstock.jpg", roi_right_3_shimstock)
    
    # 6. 시각적 확인을 위한 디버깅 이미지 생성
    debug_upper = roi_content_area.copy()
    # 세로 구분선 (파란색)
    cv2.line(debug_upper, (x_mid, 0), (x_mid, ch), (255, 0, 0), 2)
    # 좌측 가로 구분선 (녹색)
    cv2.line(debug_upper, (0, y_left_sep), (x_mid, y_left_sep), (0, 255, 0), 2)
    # 우측 가로 구분선 (빨간색)
    cv2.line(debug_upper, (x_mid, y_right_top_sep), (cw, y_right_top_sep), (0, 0, 255), 2)
    cv2.line(debug_upper, (x_mid, y_right_bottom_sep), (cw, y_right_bottom_sep), (0, 0, 255), 2)
    
    cv2.imwrite("debug_upper_refined_split.jpg", debug_upper)
    print("Upper 영역 정밀 6분할 및 저장 완료.")

# --- 실행 부분 ---
# 이전에 생성된 'roi_upper_area.jpg' 또는 메모리 상의 roi_upper 객체를 사용합니다.
split_upper_precision(roi_upper)

Upper 영역 정밀 6분할 및 저장 완료.
